# Part 1: Preprocessing

In [6]:
import pandas as pd
import random

#preprocess data
data = pd.DataFrame()
data = pd.read_csv('news_politics.csv', index_col=False)
data = data.dropna()
data = data.drop_duplicates(subset=['0'])
data = data[['0', '5']]
    
users = data['5'].unique().tolist() #1232 unique users
eligible = dict()
comment_list = []

for user in users:
    user_comments = data.loc[data['5'] == user]
    comment_list.append(user_comments)
    len_of_comments = len(user_comments)
    
    if len_of_comments >= 5:
        eligible[user] = user_comments['0'].to_list()
    
#pick random users
if len(eligible) >= 5:
    random_users = random.sample(list(eligible.keys()), 5)


# Part 2: Using Langchain

In [19]:
import getpass
import os
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
import json

api_key = os.environ["OPENAI_API_KEY"] = getpass.getpass(prompt='Enter your OpenAI API key: ')
llm = ChatOpenAI(model="gpt-4o", api_key=api_key)

template = """Question: Based on the following comments, is the user more likely a Democrat or a Republican?\n\n{comments}

Answer: Let's think step by step."""

prompt = PromptTemplate.from_template(template)

llm_chain = prompt | llm

def determine_party(comments):
    input_text = " ".join(comments)
    result = llm_chain.invoke({"comments": input_text})
    return result.content.strip()

results = []
for user in random_users:
    comments = eligible[user]
    political_party = determine_party(comments)
    results.append({
        "user": user,
        "comments": comments,
        "political_party": political_party
    })

with open('political.json', 'w') as f:
    json.dump(results, f, indent=4)


Results saved to user_affiliations.json
